# 06 - SHAP Analysis

Interpret model predictions using SHAP values.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import polars as pl
from sklearn.preprocessing import StandardScaler
import shap

from src.models.baseline import build_random_forest
from src.models.train import train_sklearn_model
from src.explain.shap_analysis import explain_with_shap

In [ ]:
df_feat = pl.read_parquet('../data/processed/hourly_demand.parquet')
feature_cols = [c for c in df_feat.columns if c not in ('pickup_hour', 'PULocationID', 'demand', 'pickup_zone', 'borough')]
df_feat = df_feat.drop_nulls().sort('pickup_hour')
n = df_feat.height
X = df_feat.select(feature_cols).to_numpy()
y = df_feat.select('demand').to_numpy().ravel()

train_end, val_end = int(n * 0.7), int(n * 0.85)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X[:train_end])
X_test_s = scaler.transform(X[val_end:])
y_train, y_test = y[:train_end], y[val_end:]

rf = train_sklearn_model(build_random_forest(), X_train_s, y_train)

In [ ]:
explain_with_shap(rf, X_train_s, X_test_s, 'RandomForest', feature_cols, Path('../outputs/figures'))
print('SHAP plots saved')

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test_s[:100])
shap.summary_plot(shap_values, X_test_s[:100], feature_names=feature_cols)